# Jetson: SFace → fixed-point encoding → masking → export

This notebook benchmarks **normal-world SFace inference**. Real OP-TEE masking is the default and requires configuring your Jetson. Missing TEE support fails without software fallback. Neither mode performs inference inside the TEE. See `README.md` for installation, model provenance, energy scope and OP-TEE setup.

The output is a masked vector, timings and optional power measurements. Raw features and random pads are not displayed or saved. This is not a full two-party verification session.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone

ROOT = Path.cwd()
if not (ROOT / "benchmark.py").exists():
    ROOT = ROOT / "jetson_benchmark"
if not (ROOT / "benchmark.py").exists():
    raise RuntimeError("Open this notebook from jetson_benchmark or its parent directory")
sys.path.insert(0, str(ROOT.resolve()))
from benchmark import SFaceSource, ReplaySource, run_benchmark, device_info
from masking import SoftwareMasker, OpteeMasker

device_info()

## Experiment settings

Use a single-face image. Detection and alignment happen once before the repeated feature benchmark. `MODE="replay"` is a local masking-only smoke test using existing data; it does **not** measure extraction. For a software-only replay test on a laptop, explicitly set `MASK_BACKEND="software"` and `MEASURE_POWER=False`. This does not satisfy the real-TEE experiment.

In [ ]:
MODE = "sface"                # "sface" or "replay"
BACKEND = "cpu"               # "cpu" or "cuda" (requires CUDA DNN OpenCV)
MASK_BACKEND = "optee"        # Real TEE required; "software" is an explicit baseline only
IMAGE = ROOT / "images/face.jpg"
ALREADY_ALIGNED = False        # True only for landmark-aligned 112x112 crops
MODEL = ROOT / "models/face_recognition_sface_2021dec.onnx"
DETECTOR = ROOT / "models/face_detection_yunet_2023mar.onnx"
EMBEDDINGS = ROOT.parent / "data/embeddings_droneface_raw_sface11.txt"

MEASURE_POWER = True
POWER_RAIL = "VDD_IN"          # Inspect your board's tegrastats output
TEGRASTATS = "tegrastats"      # Or absolute path to executable
INTERVAL_MS = 100
ITERATIONS = 100
SECONDS = 30
WARMUP = 10
NOTES = "Fill in JetPack, nvpmodel mode, clock policy, cooling and other workloads"


## Run

Energy integrates the selected rail over the repeated loop, including idle power and measurement overhead. It is not process-only energy. A missing sensor or insufficient samples fails explicitly. `MEASURE_POWER=False` produces timing results with no energy value. Each run gets a new directory.

In [ ]:
if MODE == "sface":
    source = SFaceSource(IMAGE, MODEL, DETECTOR, backend=BACKEND, aligned=ALREADY_ALIGNED)
elif MODE == "replay":
    source = ReplaySource(EMBEDDINGS)
else:
    raise ValueError("MODE must be sface or replay")

if MASK_BACKEND == "software":
    masker = SoftwareMasker()
elif MASK_BACKEND == "optee":
    masker = OpteeMasker((ROOT / "optee/host/libobscura_tee.so").resolve())
else:
    raise ValueError("MASK_BACKEND must be software or optee")

OUTPUT = ROOT / "results" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
summary = run_benchmark(
    source, masker, iterations=ITERATIONS, seconds=SECONDS, warmup=WARMUP,
    power=MEASURE_POWER, rail=POWER_RAIL, interval_ms=INTERVAL_MS,
    tegrastats=TEGRASTATS, output=OUTPUT, notes=NOTES,
)
summary

## Inspect exported packet metadata

Only the final masked embedding is exported. Its 64-bit coordinates are decimal strings to avoid JSON precision loss. Pads are discarded; using this output in the full Obscura protocol requires fresh matching correlated preprocessing. This file lives in normal-world storage even when masking used OP-TEE.

In [ ]:
import json
packet = json.loads((OUTPUT / "masked_embedding.json").read_text())
assert len(packet["values"]) == 128
assert all(0 <= int(v) < 2**64 for v in packet["values"])
print("Results:", OUTPUT)
print({k: v for k, v in packet.items() if k != "values"})
print("Timing summary (ms):", summary["timings_ms"])
print("Energy:", summary["power"])